# OncoEvidence AI — free Colab full-text appraisal

Hybrid workflow: export a search from the website, run this notebook interactively, then import the result JSON back into OncoEvidence AI. Model inference runs locally in this Colab runtime (no Groq/API inference key). Full-text retrieval is limited to Europe PMC reusable open-access XML. Checkpoints are written to Google Drive after each completed paper.

**Quality safeguards:** section-wise full-text review (not abstract-only); outcome-level classifications; design-appropriate appraisal framework; exact-quote verification against the retrieved article; unknown/unclear when evidence is absent. AI risk-of-bias ratings are preliminary and require human verification against the official tool.

**Free-tier reality:** GPU availability and runtime duration are not guaranteed. Colab says free sessions can run for at most 12 hours and may terminate earlier. Re-run the notebook and choose the same Drive checkpoint folder to resume. [Official Colab limits](https://research.google.com/colaboratory/faq.html).


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece lxml requests tqdm


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import json, os, re, time, hashlib, textwrap, requests, torch
from lxml import etree
from tqdm.auto import tqdm

# Upload the JSON exported from OncoEvidence AI (or reuse a file already in Drive).
uploaded = files.upload()
assert uploaded, 'Upload the OncoEvidence AI search export JSON.'
upload_name = next(iter(uploaded))
with open(upload_name, 'r', encoding='utf-8') as f: SEARCH_EXPORT = json.load(f)
assert isinstance(SEARCH_EXPORT.get('records'), list), 'Invalid export: records[] is missing.'

CHECKPOINT_DIR = Path('/content/drive/MyDrive/OncoEvidenceAI/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path('/content/drive/MyDrive/OncoEvidenceAI/exports')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Papers in search:', len(SEARCH_EXPORT['records']))
print('Checkpoint folder:', CHECKPOINT_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available — switch Colab runtime to GPU')


In [ ]:
# Quality-first default. 4-bit loading is intended for a free T4-class GPU when available.
# You can change MODEL_ID, but use an instruction-tuned model with adequate context length.
MODEL_ID = 'Qwen/Qwen3-8B'
MAX_INPUT_CHARS_PER_CHUNK = 6500
MAX_NEW_TOKENS_CHUNK = 900
MAX_NEW_TOKENS_FINAL = 2600

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is assigned. In Colab choose Runtime → Change runtime type → T4 GPU (if available), then rerun.')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config, device_map='auto', trust_remote_code=True)
model.eval()
print('Loaded', MODEL_ID, '| GPU memory allocated:', round(torch.cuda.memory_allocated()/1024**3, 2), 'GiB')


In [ ]:
EPMC='https://www.ebi.ac.uk/europepmc/webservices/rest'
HEADERS={'User-Agent':'OncoEvidenceAI-Colab/1.0 (full-text evidence appraisal)'}
def tidy(x): return re.sub(r'\\s+', ' ', str(x or '')).strip()
def record_key(r):
    for k in ('pmcid','pmid','doi','title'):
        if tidy(r.get(k)): return k.upper()+':'+tidy(r[k]).lower()
    return 'TITLE:unknown'
def resolve_pmc(r):
    queries=[]
    if r.get('pmcid'): queries.append(('PMCID:'+tidy(r['pmcid']).upper().replace('PMC','',1) if not tidy(r['pmcid']).upper().startswith('PMC') else 'PMCID:'+tidy(r['pmcid']).upper(), 'pmcid'))
    if r.get('pmid'): queries.append(('EXT_ID:'+tidy(r['pmid']), 'pmid'))
    if r.get('doi'): queries.append(('DOI:\"'+tidy(r['doi']).replace('\"','')+'\"','doi'))
    for q,kind in queries:
        z=requests.get(EPMC+'/search',params={'query':q,'format':'json','resultType':'core','pageSize':5},headers=HEADERS,timeout=30); z.raise_for_status()
        for row in z.json().get('resultList',{}).get('result',[]):
            if kind=='pmcid' and tidy(row.get('pmcid')).upper()==q.split(':',1)[1]: return row
            if kind=='pmid' and tidy(row.get('pmid'))==tidy(r['pmid']): return row
            if kind=='doi' and tidy(row.get('doi')).lower()==tidy(r['doi']).lower(): return row
    return None
def parse_fulltext(xml):
    root=etree.fromstring(xml.encode() if isinstance(xml,str) else xml)
    def txt(node): return tidy(' '.join(node.itertext()))
    titles=root.xpath('.//*[local-name()="article-title"]'); abstracts=root.xpath('.//*[local-name()="abstract"]')
    title=txt(titles[0]) if titles else ''
    abstract=' '.join(txt(x) for x in abstracts)
    sections=[]
    def walk(sec, parents=''):
        ts=sec.xpath('./*[local-name()="title"]'); heading=txt(ts[0]) if ts else 'Untitled section'
        label=(parents+' > '+heading).strip(' >')
        bits=[]
        for child in sec:
            tag=etree.QName(child).localname.lower()
            if tag=='sec': walk(child,label)
            elif tag in ('p','table-wrap','fig'):
                v=txt(child)
                if v: bits.append(('[TABLE] ' if tag=='table-wrap' else '[FIGURE] ' if tag=='fig' else '')+v)
        if bits: sections.append({'section':label,'text':'\\n'.join(bits)})
    for body in root.xpath('.//*[local-name()="body"]'):
        for sec in body.xpath('./*[local-name()="sec"]'): walk(sec)
        for node in body.xpath('./*[local-name()="table-wrap" or local-name()="fig"]'):
            v=txt(node)
            if v: sections.append({'section':'Body tables/figures','text':v})
    licenses=root.xpath('.//*[local-name()="license"]'); lic=txt(licenses[0]) if licenses else ''
    if licenses and licenses[0].get('{http://www.w3.org/1999/xlink}href'): lic+=' ('+licenses[0].get('{http://www.w3.org/1999/xlink}href')+')'
    return {'title':title,'abstract':abstract,'sections':sections,'license':lic}
DOMAINS={
 'RoB 2':['Randomization process','Deviations from intended interventions','Missing outcome data','Outcome measurement','Selection of reported result'],
 'ROBINS-I':['Confounding','Participant selection','Intervention classification','Deviations from intended interventions','Missing data','Outcome measurement','Selective reporting'],
 'PROBAST+AI':['Participants and data sources','Predictors','Outcome','Analysis','Applicability'],
 'PROBAST':['Participants and data sources','Predictors','Outcome','Analysis','Applicability'],
 'QUADAS-2':['Patient selection','Index test','Reference standard','Flow and timing','Applicability'],
 'AMSTAR 2':['Protocol','Search and selection','Risk-of-bias assessment','Synthesis methods','Publication bias','Conflicts of interest'],
 'AGREE II':['Scope and purpose','Stakeholder involvement','Rigor of development','Clarity','Applicability','Editorial independence'],
 'SANRA':['Importance','Aims','Literature search','Referencing','Scientific reasoning','Data presentation']}
def framework_for(r):
    supplied=tidy(r.get('recommended_appraisal_tool'))
    if supplied: return supplied
    d=(tidy(r.get('study_design'))+' '+ ' '.join(r.get('publication_types') or [])).lower()
    if 'random' in d or 'clinical trial' in d: return 'RoB 2'
    if any(x in d for x in ('cohort','case-control','observational','cross-sectional')): return 'ROBINS-I'
    if any(x in d for x in ('diagnostic','accuracy')): return 'QUADAS-2'
    if any(x in d for x in ('systematic review','meta-analysis')): return 'AMSTAR 2'
    if any(x in d for x in ('prediction','prognostic','machine learning','radiomics')): return 'PROBAST+AI'
    return 'JBI appropriate checklist (select subtype during human verification)'
def split_sections(sections, limit=MAX_INPUT_CHARS_PER_CHUNK):
    chunks=[]
    for sec in sections:
        text=sec['text']
        paras=re.split(r'(?<=[.!?])\\s+(?=[A-Z0-9])',text)
        buf=''
        for p in paras:
            if len(buf)+len(p)+1>limit and buf: chunks.append({'section':sec['section'],'text':buf}); buf=p
            else: buf=(buf+' '+p).strip()
        if buf: chunks.append({'section':sec['section'],'text':buf})
    return chunks
def extract_json(s):
    s=re.sub(r'^```(?:json)?|```$','',s.strip(),flags=re.I|re.M).strip()
    a=s.find('{'); b=s.rfind('}')
    if a<0 or b<=a: raise ValueError('Model did not return a JSON object: '+s[:300])
    return json.loads(s[a:b+1])
def generate_json(system,user,max_tokens):
    messages=[{'role':'system','content':system},{'role':'user','content':user+'\\n/no_think'}]
    prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=tokenizer.model_max_length if tokenizer.model_max_length<100000 else 12000).to(model.device)
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=max_tokens,do_sample=False,temperature=None,top_p=None,pad_token_id=tokenizer.eos_token_id)
    answer=tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True)
    return extract_json(answer)
def exact_quote_check(obj, source):
    src=tidy(source).lower()
    def clean_quotes(items):
        out=[]
        for q in items or []:
            excerpt=tidy(q.get('excerpt','')) if isinstance(q,dict) else ''
            if excerpt and excerpt.lower() in src: out.append({**q,'excerpt':excerpt[:500]})
        return out
    for o in obj.get('outcomes',[]): o['evidence_quotes']=clean_quotes(o.get('evidence_quotes'))
    risk=obj.get('risk_of_bias') or {}
    for d in risk.get('domains',[]): d['evidence_quotes']=clean_quotes(d.get('evidence_quotes'))
    return obj
def save_checkpoint(key, payload):
    fn=hashlib.sha256(key.encode()).hexdigest()+'.json'
    tmp=CHECKPOINT_DIR/(fn+'.tmp'); dest=CHECKPOINT_DIR/fn
    tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8'); tmp.replace(dest)
def load_checkpoints():
    found={}
    for p in CHECKPOINT_DIR.glob('*.json'):
        try:
            obj=json.loads(p.read_text(encoding='utf-8')); found[obj.get('record_key')]=obj
        except Exception: pass
    return found


In [ ]:
SYSTEM='''You are a meticulous clinical evidence reviewer. Use only the supplied article text. Never invent outcomes, estimates, confidence intervals, p-values, sample sizes, or quotes. Classify EACH outcome separately as benefit, no_clear_difference, harm, mixed, unclear, or not_reported. A non-significant p-value is not proof of equivalence. Separate statistical significance from clinical importance. Preserve comparator, endpoint direction, timepoint, denominator and effect measure. For risk of bias, use only the specified design-appropriate framework domains; do not infer missing methods. Use unclear when information is insufficient. Return one valid JSON object only.'''
CHUNK_SCHEMA='''{"observations":[{"outcome":"","outcome_group":"benefit|no_clear_difference|harm|mixed|unclear|not_reported","endpoint":"","timepoint":"","population_n":"","events_or_summary":"","effect_measure":"","effect_estimate":"","confidence_interval":"","p_value":"","statistical_significance":"significant|not significant|not reported","clinical_interpretation":"","evidence_quotes":[{"section":"","excerpt":"","location":""}]}],"methods_evidence":"","limitations_reported":[],"funding_and_conflicts":"","risk_of_bias_evidence":[{"domain":"","judgement":"low|some_concerns|high|unclear|not_applicable","rationale":"","evidence_quotes":[{"section":"","excerpt":""}]}],"warnings":[]}'''
FINAL_SCHEMA='''{"study_design_verified":"","population":"","intervention":"","comparator":"","follow_up":"","outcomes":[{"outcome":"","outcome_group":"benefit|no_clear_difference|harm|mixed|unclear|not_reported","endpoint":"","timepoint":"","population_n":"","events_or_summary":"","effect_measure":"","effect_estimate":"","confidence_interval":"","p_value":"","statistical_significance":"significant|not significant|not reported","clinical_interpretation":"","evidence_quotes":[{"section":"","excerpt":"","location":""}]}],"main_findings":"","methods_assessment":"","limitations_reported":[],"funding_and_conflicts":"","risk_of_bias":{"overall_judgement":"low|some_concerns|high|unclear","domains":[{"domain":"","judgement":"low|some_concerns|high|unclear|not_applicable","rationale":"","evidence_quotes":[{"section":"","excerpt":""}]}]},"appraisal_warnings":[]}'''
def appraise_paper(record):
    meta=resolve_pmc(record)
    if not meta: return {'status':'full_text_unavailable','reason':'No matching Europe PMC full-text record found.'}
    pmcid=tidy(meta.get('pmcid')).upper()
    if meta.get('isOpenAccess')!='Y' or meta.get('inEPMC')!='Y': return {'status':'full_text_unavailable','reason':'Europe PMC does not mark this paper as reusable open-access full text.','pmcid':pmcid}
    resp=requests.get(f'{EPMC}/{pmcid}/fullTextXML',headers=HEADERS,timeout=60)
    if resp.status_code==404: return {'status':'full_text_unavailable','reason':'Reusable full-text XML is not exposed for this paper.','pmcid':pmcid}
    resp.raise_for_status(); full=parse_fulltext(resp.text)
    if sum(len(x['text']) for x in full['sections'])<1200: return {'status':'full_text_incomplete','reason':'Retrieved full text is too short for appraisal.','pmcid':pmcid}
    tool=framework_for(record); domains=next((v for k,v in DOMAINS.items() if k.lower() in tool.lower()),['Participant selection','Intervention or exposure measurement','Outcome measurement','Confounding and follow-up','Statistical analysis'])
    chunks=split_sections(full['sections'])
    extracted=[]
    for i,ch in enumerate(chunks):
        user=f'''Paper: {record.get('title','')}\\nStudy identifier: {record.get('pmid','')} / {pmcid}\\nFramework: {tool}\\nRequired RoB domains: {domains}\\nThis is article section {i+1}/{len(chunks)}: {ch['section']}\\nExtract outcome-specific numerical results and only the methods/RoB evidence actually present in this section. Do not make an overall paper judgement from one section. If nothing relevant is present, return empty arrays.\\nJSON schema: {CHUNK_SCHEMA}\\nARTICLE TEXT:\\n{ch['text']}'''
        extracted.append(generate_json(SYSTEM,user,MAX_NEW_TOKENS_CHUNK))
    evidence=json.dumps(extracted,ensure_ascii=False)
    final_user=f'''Paper metadata: {json.dumps({k:record.get(k) for k in ['title','journal','year','pmid','doi','pmcid']},ensure_ascii=False)}\\nAppraisal framework: {tool}\\nDomains to assess: {domains}\\nYou are given section-by-section evidence extracted from the ENTIRE accessible article. Synthesize without inventing. Merge duplicate outcomes, but retain distinct endpoints/timepoints. If an outcome has conflicting results across timepoints, classify as mixed and explain. For each RoB domain give a conservative judgement based on the evidence; use unclear if evidence is absent. Overall judgement must reflect domain judgements and study design. Do not claim a formal validated assessment.\\nJSON schema: {FINAL_SCHEMA}\\nSECTION EVIDENCE:\\n{evidence}'''
    data=generate_json(SYSTEM,final_user,MAX_NEW_TOKENS_FINAL)
    source=full['title']+' '+full['abstract']+' '+' '.join(x['text'] for x in full['sections'])
    data=exact_quote_check(data,source)
    allowed={'benefit','no_clear_difference','harm','mixed','unclear','not_reported'}
    outcomes=[]
    for o in data.get('outcomes',[])[:20]:
        if not isinstance(o,dict): continue
        if o.get('outcome_group') not in allowed: o['outcome_group']='unclear'
        o['evidence_quotes']=(o.get('evidence_quotes') or [])[:5]; outcomes.append(o)
    risk=data.get('risk_of_bias') or {}; judgements={'low','some_concerns','high','unclear','not_applicable'}
    rb=[]
    for d in risk.get('domains',[])[:12]:
        if not isinstance(d,dict): continue
        if d.get('judgement') not in judgements: d['judgement']='unclear'
        d['evidence_quotes']=(d.get('evidence_quotes') or [])[:4]; rb.append(d)
    if len(rb)<2: risk['overall_judgement']='unclear'
    risk['domains']=rb; risk['tool']=tool; risk['note']='AI-assisted preliminary domain judgements from retrieved open-access full text; verify against the official instrument and complete article.'
    appraisal={**data,'outcomes':outcomes,'outcome_groups':sorted(set(o['outcome_group'] for o in outcomes)) or ['unclear'],'risk_of_bias':risk,
      'retrieved_from':'Europe PMC fullTextXML','pmcid':pmcid,'license':full['license'],'model':MODEL_ID,
      'evidence_integrity':{'verified_excerpt_count':sum(len(o.get('evidence_quotes',[])) for o in outcomes)+sum(len(d.get('evidence_quotes',[])) for d in rb),'unverified_quotes_removed':True,'full_text_characters_reviewed':sum(len(x['text']) for x in full['sections']),'sections_reviewed':len(full['sections'])},
      'appraisal_warnings':(data.get('appraisal_warnings') or [])+(['No outcome extracted; verify whether the article reports eligible outcomes.'] if not outcomes else [])}
    return {'status':'appraised','pmcid':pmcid,'appraisal':appraisal}


In [ ]:
completed=load_checkpoints()
records=SEARCH_EXPORT['records']
pending=[r for r in records if record_key(r) not in completed]
print(f'{len(completed)} checkpointed · {len(pending)} remaining · {len(records)} total')
errors=[]
for rec in tqdm(pending,desc='Full-text appraisals'):
    key=record_key(rec)
    try:
        payload=appraise_paper(rec)
        row={'record_key':key,'record':rec,'full_text_appraisal':payload}
        save_checkpoint(key,row); completed[key]=row
    except Exception as e:
        errors.append({'record_key':key,'title':rec.get('title'),'error':repr(e)})
        # Save the failure as a checkpoint so the error is visible; remove its checkpoint to retry later.
        print('ERROR:',rec.get('title'),repr(e))
    # Avoid hammering Europe PMC; local inference has no API rate limit.
    time.sleep(0.5)
print('Completed/checkpointed:',len(completed),'Errors:',len(errors))


In [ ]:
# Build import-compatible result JSON. Existing website appraisals are preserved if Colab has no newer result.
merged=[]
for rec in records:
    row=completed.get(record_key(rec))
    if row:
        rec_out={**rec,'full_text_appraisal':row['full_text_appraisal']}
        if row['full_text_appraisal'].get('pmcid'): rec_out['pmcid']=row['full_text_appraisal']['pmcid']
    else: rec_out=rec
    merged.append(rec_out)
result={**SEARCH_EXPORT,'records':merged,'inference_engine':'Google Colab local inference','model':MODEL_ID,'colab_errors':errors}
out_path=EXPORT_DIR/'oncoevidence_colab_results.json'
out_path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:',out_path,'| appraised:',sum(1 for r in merged if r.get('full_text_appraisal',{}).get('status')=='appraised'))
files.download(str(out_path))


## Human verification checklist
- Confirm study design and the selected instrument/subtype.
- Verify every outcome's comparator, endpoint direction, denominator, timepoint, estimate and CI in the paper.
- Verify each domain judgement against the official RoB 2 / ROBINS-I / QUADAS-2 / PROBAST+AI / AMSTAR 2 guidance.
- AI-generated judgements are not a substitute for independent reviewer assessment or consensus.
- Only Europe PMC reusable OA full-text XML is processed here; inaccessible papers remain explicitly separated.
